# Task 1 — Rule-Based Baseline
Replicates what Suricata rules would detect using threshold logic.
Dataset: UNSW-NB15 (prepared by teammate)
Goal: measure F1, Precision, Recall → save as benchmark for ML comparison

In [19]:
import sys
!{sys.executable} -m pip install numpy pandas scikit-learn -q

import numpy as np
import pandas as pd
import json, os
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
print("Imports successful")

Imports successful


## Load the data
Loading X_test and y_test — the held-out test set.
These files were prepared by our teammate (Data Engineer, Deliverable 2).

In [24]:
DATA_DIR = "../data/processed"   # adjust this path if needed

X_test = np.load(f"{DATA_DIR}/X_test.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

with open(f"{DATA_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Features loaded: {len(FEATURE_COLS)}")
print(f"Attack rate: {y_test.mean()*100:.1f}%")

X_test shape: (82332, 37)
y_test shape: (82332,)
Features loaded: 37
Attack rate: 55.1%


## Check available feature names
We need to know the exact names before writing rules.
Print the full list and find the ones we need for each attack type.

In [28]:
# Print all features so we know exactly which names to use in rules
for i, name in enumerate(FEATURE_COLS):
    print(f"  [{i:2d}] {name}")

  [ 0] dur
  [ 1] rate
  [ 2] sload
  [ 3] dload
  [ 4] spkts
  [ 5] dpkts
  [ 6] sbytes
  [ 7] dbytes
  [ 8] sloss
  [ 9] dloss
  [10] sinpkt
  [11] dinpkt
  [12] sjit
  [13] djit
  [14] swin
  [15] dwin
  [16] tcprtt
  [17] synack
  [18] ackdat
  [19] smean
  [20] dmean
  [21] trans_depth
  [22] response_body_len
  [23] ct_src_dport_ltm
  [24] ct_dst_sport_ltm
  [25] is_ftp_login
  [26] ct_ftp_cmd
  [27] ct_flw_http_mthd
  [28] is_sm_ips_ports
  [29] proto_enc
  [30] service_enc
  [31] state_enc
  [32] byte_ratio
  [33] pkt_ratio
  [34] total_bytes
  [35] jit_ratio
  [36] dur_bin_enc


## ── The rule-based baseline ───────────────────────────────────────
This replicates what Suricata rules would detect using threshold logic.
Each rule corresponds to a real Suricata detection pattern.
On SCALED data (mean=0, std=1):
positive values = above average for that feature
negative values = below average for that feature

In [31]:
def idx(name):
    """Get column index for a feature name."""
    return FEATURE_COLS.index(name)
    
def apply_suricata_rules(X):
    """
    Simulate Suricata rule logic on pre-scaled UNSW-NB15 features.
    Returns: array of 0 (normal) or 1 (attack flagged)
    """
    n = len(X)
    predictions = np.zeros(n)   # start: everything is normal

    # ── RULE 1: C2 Beaconing ──────────────────────────────────
    # Suricata rule: same src->dst >10 times in 60s
    # Feature proxy: sjit (source jitter) — C2 traffic has
    # unnaturally LOW jitter because it beacons like clockwork.
    # After scaling: very low sjit = well below average = < -0.5
    c2_mask = X[:, idx('sjit')] < -0.5
    predictions[c2_mask] = 1
    print(f"Rule 1 (C2 Beaconing):       {c2_mask.sum():,} flows flagged")

    # ── RULE 2: Lateral Movement / Port Scan ──────────────────
    # Suricata rule: >30 SYN packets from one source in 10s
    # Feature proxy: high spkts (source packets) AND very short
    # duration — scanning sends many packets very quickly
    scan_mask = (X[:, idx('spkts')] > 1.5) & (X[:, idx('dur')] < -0.4)
    predictions[scan_mask] = 1
    print(f"Rule 2 (Lateral Movement):   {scan_mask.sum():,} flows flagged")

    # ── RULE 3: Data Exfiltration ─────────────────────────────
    # Suricata rule: large outbound bytes in a single session
    # Feature proxy: byte_ratio — much more sent than received.
    # Exfiltration sends data out (high sbytes) but receives little (low dbytes)
    exfil_mask = X[:, idx('byte_ratio')] > 2.5
    predictions[exfil_mask] = 1
    print(f"Rule 3 (Exfiltration):       {exfil_mask.sum():,} flows flagged")

    # ── RULE 4: DDoS / Flood ──────────────────────────────────
    # Suricata rule: abnormally high traffic rate
    # Feature proxy: rate (packets per second) very high
    ddos_mask = X[:, idx('rate')] > 3.0
    predictions[ddos_mask] = 1
    print(f"Rule 4 (DDoS/Flood):         {ddos_mask.sum():,} flows flagged")


    # ── RULE 5: DNS Tunnelling ────────────────────────────────
    # Suricata rule: DNS query names > 100 chars
    # Feature proxy: high source load (sload) with high ct_flw_http_mthd
    # (abnormal HTTP method counts = protocol abuse typical of tunnelling)
    tunnel_mask = (X[:, idx('sload')] > 2.0) & \
                  (X[:, idx('ct_flw_http_mthd')] > 1.5)
    predictions[tunnel_mask] = 1
    print(f"Rule 5 (DNS Tunnelling):     {tunnel_mask.sum():,} flows flagged")

    return predictions

print("=" * 55)
print("Running rule-based baseline on test data...")
print("=" * 55)
y_pred_rules = apply_suricata_rules(X_test)

Running rule-based baseline on test data...
Rule 1 (C2 Beaconing):       0 flows flagged
Rule 2 (Lateral Movement):   0 flows flagged
Rule 3 (Exfiltration):       309 flows flagged
Rule 4 (DDoS/Flood):         1,036 flows flagged
Rule 5 (DNS Tunnelling):     0 flows flagged


## Measure and record the baseline metrics
These four numbers define the performance of traditional rule-based detection.
The ML model in Task 2 must beat the F1 score here by at least 0.15.

In [41]:
# ── Calculate and display all metrics ────────────────────────────
baseline_f1  = f1_score(y_test, y_pred_rules)
baseline_pre = precision_score(y_test, y_pred_rules)
baseline_rec = recall_score(y_test, y_pred_rules)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rules).ravel()
baseline_fpr = fp / (fp + tn)


print("\n" + "=" * 55)
print("RULE-BASED BASELINE — FINAL METRICS")
print("=" * 55)
print(f"F1 Score:            {baseline_f1:.4f}")
print(f"Precision:           {baseline_pre:.4f}  (when it fires, how often right?)")
print(f"Recall:              {baseline_rec:.4f}  (what % of attacks did it catch?)")
print(f"False Positive Rate: {baseline_fpr:.4f}  (how much noise?)")
print(f"\nConfusion Matrix:")
print(f"  Caught attacks (TP):          {tp:,}")
print(f"  Missed attacks (FN):          {fn:,}  ← rules miss these")
print(f"  False alarms (FP):            {fp:,}  ← noise / alert fatigue")
print(f"  Correctly ignored benign (TN):{tn:,}")
print(f"\nML model must achieve F1 > {baseline_f1 + 0.15:.4f} to meet 15% target")


RULE-BASED BASELINE — FINAL METRICS
F1 Score:            0.0472
Precision:           0.8193  (when it fires, how often right?)
Recall:              0.0243  (what % of attacks did it catch?)
False Positive Rate: 0.0066  (how much noise?)

Confusion Matrix:
  Caught attacks (TP):          1,102
  Missed attacks (FN):          44,230  ← rules miss these
  False alarms (FP):            243  ← noise / alert fatigue
  Correctly ignored benign (TN):36,757

ML model must achieve F1 > 0.1972 to meet 15% target


In [ ]:
os.makedirs("models/baseline", exist_ok=True)
metrics = {"f1":baseline_f1,"precision":baseline_pre,"recall":baseline_rec,"fpr":baseline_fpr,
           "tp":int(tp),"fp":int(fp),"fn":int(fn),"tn":int(tn)}
with open("models/baseline/baseline_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print("Saved to models/baseline/baseline_metrics.json")

Saved to models/baseline/baseline_metrics.json
